# Day 2：简单线性回归

目标：从学习时长预测分数，理解最小二乘、斜率和残差。使用 studentscores.csv 的全部 28 行；包含偏离时长与分数整体趋势的样本，保留原数据，不在未经核实的情况下删除。分数与学习时长的相关关系不等于因果。

运行前请阅读[环境与运行说明](../docs/setup.md)。本课 `.py` 是教学源文件，配套 Markdown 和 Notebook 自动同步。图形保存到 `outputs/`，设置 `COURSE_SHOW_PLOTS=1` 可显示窗口。


[Python 源文件](Day%202_Simple_Linear_Regression.py) · [Notebook](Day%202_Simple_Linear_Regression.ipynb) · [完整课程目录](../docs/curriculum.md)


In [ ]:
from pathlib import Path
import sys

# 脚本从文件位置定位仓库；Notebook 从当前工作目录向上查找。
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for candidate in (base, *base.parents):
    if (candidate / "Code" / "course_utils.py").is_file():
        code_dir = str(candidate / "Code")
        if code_dir not in sys.path:
            sys.path.insert(0, code_dir)
        break
else:
    raise FileNotFoundError("找不到课程仓库，请从仓库根目录或 Code 目录启动 Notebook。")
from course_utils import DATA, OUTPUT, finish_plot


## 读取数据并保留测试集

本课模型为 ŷ=b+w×Hours，通过最小化训练残差平方和学习 w、b。X 必须是二维表（n,1），y 是一维目标（n,）。固定划分便于比较；很小的测试集会使指标波动明显。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from course_utils import regression_report

data = pd.read_csv(DATA / "studentscores.csv")
X, y = data[["Hours"]], data["Scores"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Slope:", model.coef_[0], "Intercept:", model.intercept_)
metrics = regression_report(y_test, y_pred)


## 解读误差和图形

MAE 是平均绝对误差，RMSE 对大误差更敏感，单位均是分数；R²=1−Σ(y−ŷ)²/Σ(y−ȳ)²，其中 ȳ 是当前评价集合的真实均值，可为负。它的分母基准不同于用训练集均值建立的 DummyRegressor。训练/测试图分开，回归线用排序后的时长绘制，残差图观察系统性偏差。


In [ ]:
for label, features, targets in [("train", X_train, y_train), ("test", X_test, y_test)]:
    fig, ax = plt.subplots()
    ax.scatter(features["Hours"], targets, label="Observed")
    ordered = features.sort_values("Hours")
    ax.plot(ordered["Hours"], model.predict(ordered), color="red", label="Predicted")
    ax.set(xlabel="Study hours", ylabel="Score", title=label)
    ax.legend()
    finish_plot("day02_" + label)
fig, ax = plt.subplots()
ax.scatter(y_pred, y_test.to_numpy() - y_pred)
ax.axhline(0, color="black")
ax.set(xlabel="Predicted score", ylabel="Residual (observed - predicted)")
finish_plot("day02_residuals")


## 练习与检查

1. 比较训练集均值基线与模型的测试 MAE。
2. 列出残差最大的样本，讨论它为何值得核实，但不能仅因拟合差而删除。
3. 改变划分种子观察指标波动；不要挑测试集表现最好的一次报告。
